# Ensemble zero-shot experiment

This notebook combines outputs from the four individual zero-shot experiments. It does not use few-shot demonstrations or training examples. It includes majority fusion, dev-weighted voting scaffolding, and an editable optional judge/refiner prompt.

Results remain blank until execution.


## Protocol

The individual models receive only complete `.txt` documents. Matching `.json` files are ground truth for evaluation. This notebook loads the individual audit files and preserves the input and annotation filenames in the ensemble result.


In [ ]:
from pathlib import Path
import json, re, xml.etree.ElementTree as ET
from collections import defaultdict, Counter

# ---------- Configuration ----------
BASE = Path("/content/experimental_setup") if Path("/content").exists() else (Path.cwd() if (Path.cwd() / "mining-legal-arguments").exists() else Path.cwd() / "experimental_setup")
DATA_ROOT = BASE / "mining-legal-arguments"
if not DATA_ROOT.exists():
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/trusthlt/mining-legal-arguments.git", str(DATA_ROOT)], check=True)
PREPARED_ROOT = BASE / "processed_echr"
OUTPUT_DIR = BASE / "outputs"
MODEL_OUTPUTS = {
    "saul_7b": OUTPUT_DIR / "01_saul_7b_instruct.json",
    "saullm_54b": OUTPUT_DIR / "02_saullm_54b_instruct.json",
    "law_chat": OUTPUT_DIR / "03_law_chat.json",
    "llama3_8b": OUTPUT_DIR / "04_paper_llama3_8b_instruct.json",
}
RUN_ENSEMBLE = False

LABELS = [
    "Procedural arguments—Non contestation by the parties",
    "Method of interpretation—Textual interpretation",
    "Method of interpretation—Historical interpretation",
    "Method of interpretation—Systematic interpretation",
    "Method of interpretation—Teleological interpretation",
    "Method of interpretation—Comparative law",
    "Test of the principle of proportionality—Legal basis",
    "Test of the principle of proportionality—Legitimate purpose",
    "Test of the principle of proportionality—Suitability",
    "Test of the principle of proportionality—Necessity/Proportionality",
    "Institutional arguments—Overruling",
    "Institutional arguments—Distinguishing",
    "Institutional arguments—Margin of Appreciation",
    "Precedents of the ECHR",
    "Decision of the ECHR",
    "Application to the concrete case",
]
XMI_LABELS = {
    "Konsens der prozessualen Parteien": LABELS[0],
    "Wortlaut Auslegung": LABELS[1],
    "Historische Auslegung": LABELS[2],
    "Systematische Auslegung": LABELS[3],
    "Sinn & Zweck Auslegung": LABELS[4],
    "Rechtsvergleichung": LABELS[5],
    "Verhältnismäßigkeitsprüfung – Rechtsgrundlage": LABELS[6],
    "Verhältnismäßigkeitsprüfung – Legitimer Zweck": LABELS[7],
    "Verhältnismäßigkeitsprüfung – Geeignetheit": LABELS[8],
    "Verhältnismäßigkeitsprüfung – Angemessenheit": LABELS[9],
    "Overruling": LABELS[10], "Distinguishing": LABELS[11],
    "Einschätzungsspielraum": LABELS[12],
    "Vorherige Rechtsprechung des EGMR": LABELS[13],
    "Entscheidung des EGMR": LABELS[14], "Subsumtion": LABELS[15],
}

# ---------- Dataset export/loading ----------
def _load_xmi(path, split):
    root = ET.parse(path).getroot()
    text = next((e.attrib["sofaString"] for e in root.iter()
                 if e.tag.endswith("Sofa") and e.attrib.get("sofaString")), "")
    annotations = []
    for element in root.iter():
        if not element.tag.endswith("LegalArgumentation"):
            continue
        label = XMI_LABELS.get(" ".join(element.attrib.get("ArgType", "").split()))
        if label and element.attrib.get("begin") and element.attrib.get("end"):
            start, end = int(element.attrib["begin"]), int(element.attrib["end"])
            annotations.append({"start": start, "end": end, "text": text[start:end], "label": label})
    return {"id": path.stem, "split": split, "text": text, "annotations": annotations,
            "source": str(path)} if text and annotations else None

def load_xmi_split(data_root, split):
    repository_split = {"dev": "val"}.get(split, split)
    ids = sorted(p.stem for p in (Path(data_root) / "data" / repository_split / "agent").glob("*.csv"))
    records = []
    for doc_id in ids:
        path = Path(data_root) / "gold_data" / f"{doc_id}.xmi"
        if path.exists():
            record = _load_xmi(path, split)
            if record:
                records.append(record)
    if not records:
        raise RuntimeError(f"No XMI records found for {split}")
    return records

def prepare_echr_files(data_root=DATA_ROOT, output_root=PREPARED_ROOT):
    output_root = Path(output_root)
    manifest = []
    for split in ("train", "dev", "test"):
        text_dir, annotation_dir = output_root / split / "texts", output_root / split / "annotations"
        text_dir.mkdir(parents=True, exist_ok=True)
        annotation_dir.mkdir(parents=True, exist_ok=True)
        for record in load_xmi_split(data_root, split):
            text_file = text_dir / f"{record['id']}.txt"
            annotation_file = annotation_dir / f"{record['id']}.json"
            text_file.write_text(record["text"], encoding="utf-8")
            annotation_file.write_text(json.dumps({
                "id": record["id"], "split": split,
                "annotations": record["annotations"], "source": record["source"]
            }, ensure_ascii=False, indent=2), encoding="utf-8")
            manifest.append({"id": record["id"], "split": split,
                             "input_file": str(text_file),
                             "annotation_file": str(annotation_file)})
    (output_root / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest

def load_prepared(root=PREPARED_ROOT, split="test"):
    root = Path(root)
    records = []
    for annotation_file in sorted((root / split / "annotations").glob("*.json")):
        annotation = json.loads(annotation_file.read_text(encoding="utf-8"))
        text_file = root / split / "texts" / f"{annotation_file.stem}.txt"
        records.append({"id": annotation["id"], "split": split,
                        "text": text_file.read_text(encoding="utf-8"),
                        "annotations": annotation["annotations"],
                        "input_file": str(text_file),
                        "annotation_file": str(annotation_file)})
    if not records:
        raise RuntimeError("No prepared files found. Run prepare_echr_files first.")
    return records

MANIFEST = prepare_echr_files()
print(f"Prepared {len(MANIFEST)} documents.")


In [ ]:
# ---------- Editable zero-shot ensemble/refiner prompt ----------
# This cell is only for the optional LLM judge/refiner. It receives candidate
# predictions, not ground-truth annotations or few-shot examples.
JUDGE_SYSTEM_PROMPT = """You are a legal argument prediction verifier.
Given a document and candidate predictions from several frozen zero-shot models,
return JSON only in the required schema. Keep only spans copied exactly from the
document and assign one allowed label. Do not add relations or actors."""

def build_judge_prompt(document_text, candidates):
    allowed = "\n".join(f"- {label}" for label in LABELS)
    return f"""{JUDGE_SYSTEM_PROMPT}

Allowed labels:
{allowed}

Required JSON schema:
{{"arguments": [{{"text": "exact copied span", "label": "one allowed label"}}]}}

DOCUMENT:
{document_text}

CANDIDATE PREDICTIONS:
{json.dumps(candidates, ensure_ascii=False, indent=2)}

JSON OUTPUT:"""


In [ ]:
# ---------- Ensemble loading, fusion, and scoring ----------
def load_prediction_file(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    return data, [item.get("predictions", []) for item in data.get("audit", [])]

def majority_fusion(predictions_by_model):
    groups = []
    for predictions in predictions_by_model.values():
        for prediction in predictions:
            group = next((g for g in groups if
                          max(prediction["start"], g[0]["start"]) <
                          min(prediction["end"], g[0]["end"])), None)
            if group is None:
                groups.append([prediction])
            else:
                group.append(prediction)
    fused = []
    for group in groups:
        label = Counter(item["label"] for item in group).most_common(1)[0][0]
        chosen = next(item for item in group if item["label"] == label)
        fused.append({**chosen, "support": len(group), "method": "majority"})
    return fused

def weighted_fusion(predictions_by_model, weights):
    groups = []
    for model_name, predictions in predictions_by_model.items():
        for prediction in predictions:
            group = next((g for g in groups if
                          max(prediction["start"], g[0]["prediction"]["start"]) <
                          min(prediction["end"], g[0]["prediction"]["end"])), None)
            item = {"model": model_name, "prediction": prediction}
            if group is None:
                groups.append([item])
            else:
                group.append(item)
    fused = []
    for group in groups:
        scores = defaultdict(float)
        for item in group:
            scores[item["prediction"]["label"]] += weights.get(item["model"], 1.0)
        label = max(scores, key=scores.get)
        chosen = next(item["prediction"] for item in group
                      if item["prediction"]["label"] == label)
        confidence = scores[label] / sum(scores.values())
        fused.append({**chosen, "support": len(group), "confidence": confidence,
                      "method": "dev_weighted"})
    return fused

def match_count(predictions, gold, labeled=True, overlap=False):
    used, hits = set(), 0
    for prediction in predictions:
        for index, target in enumerate(gold):
            if index in used or (labeled and prediction["label"] != target["label"]):
                continue
            matched = (prediction["start"] == target["start"] and
                       prediction["end"] == target["end"]) if not overlap else (
                       max(prediction["start"], target["start"]) <
                       min(prediction["end"], target["end"]))
            if matched:
                used.add(index); hits += 1; break
    return hits

def score_records(records, predictions):
    totals = defaultdict(int)
    measures = {"exact_joint": (True, False), "overlap_joint": (True, True),
                "span_exact": (False, False), "span_overlap": (False, True)}
    for record, predicted in zip(records, predictions):
        for name, (labeled, overlap) in measures.items():
            hits = match_count(predicted, record["annotations"], labeled, overlap)
            totals[name+"_tp"] += hits
            totals[name+"_fp"] += len(predicted) - hits
            totals[name+"_fn"] += len(record["annotations"]) - hits
    result = {}
    for name in measures:
        tp, fp, fn = (totals[name+s] for s in ("_tp", "_fp", "_fn"))
        p, r = (tp/(tp+fp) if tp+fp else 0), (tp/(tp+fn) if tp+fn else 0)
        result[name] = {"precision": p, "recall": r, "f1": 2*p*r/(p+r) if p+r else 0}
    return result

def derive_dev_weights(dev_records, dev_predictions):
    return {name: score_records(dev_records, predictions)["exact_joint"]["f1"]
            for name, predictions in dev_predictions.items()}

def run_ensemble():
    test_records = load_prepared(PREPARED_ROOT, "test")
    loaded = {name: load_prediction_file(path) for name, path in MODEL_OUTPUTS.items()}
    model_data = {name: value[0] for name, value in loaded.items()}
    test_predictions = {name: value[1] for name, value in loaded.items()}
    majority = [majority_fusion({name: preds[i] for name, preds in test_predictions.items()})
                for i in range(len(test_records))]
    result = {"mode": "zero-shot-ensemble", "input_manifest": str(PREPARED_ROOT / "manifest.json"),
              "majority_metrics": score_records(test_records, majority),
              "experiment_record": {"date": "", "runtime_seconds": "",
                                    "dev_weights": "", "judge_model": "",
                                    "metrics": "", "errors": "", "observations": ""},
              "input_files": [record["input_file"] for record in test_records],
              "annotation_files": [record["annotation_file"] for record in test_records]}
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    target = OUTPUT_DIR / "05_ensemble_results.json"
    target.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result["majority_metrics"], indent=2))
    print(f"Saved: {target}")
    return result

if RUN_ENSEMBLE:
    result = run_ensemble()
